# 03 · Interpretability: PARNET's additive mix-coefficient per RBP

**What is being tested.** PARNET predicts each profile as an additive mixture of a sequence-conditioned
**target** channel and an RNA-only **control** (bias) channel, weighted by a learned per-RBP
**mix-coefficient**. We read this coefficient across RBPs and families.

**Why test it.** The mix-coefficient is the interpretability handle that tells us *for which RBPs the
profile is driven by sequence* (high mix → sequence-specific, the regime where conditioning on a protein
representation can plausibly add value) *vs which are bias-dominated* (low mix). It directly feeds the
leave-out group-selection deliverable (which RBPs to test vs use as falsification controls).

**Reasoning.** mix_coeff near 1 ⇒ the predicted shape comes from the sequence-driven target channel;
near 0 ⇒ from the RNA-only background. Motif-driven RBPs should sit high; non-motif/structural RBPs low.

## Definition: the mix-coefficient

From the `AdditiveMix` head, $p_{\mathrm{tot}}=\pi\,p_{\mathrm{tgt}}+(1-\pi)\,p_{\mathrm{ctrl}}$ with a
learned per-track **mix-coefficient** $\pi=\sigma(\texttt{mix\_logit})\in(0,1)$. $\pi\to 1$ means the
predicted profile is essentially the sequence-conditioned **target** channel (sequence-driven); $\pi\to 0$
means it is the RNA-only **control** background (bias-driven). We read $\pi$ per RBP track as an
interpretability handle: high-$\pi$, motif-bearing RBPs are where conditioning on a protein representation
can plausibly add value.

In [ ]:
import os, sys, json, pathlib
import numpy as np
_here = pathlib.Path.cwd().resolve()
REPO = next((c for c in (_here, *_here.parents) if (c / "src" / "mmpartnet").is_dir()), _here)
sys.path.insert(0, str(REPO / "src"))
from mmpartnet import config
COMMITTED = REPO / "mmpartnet_out"          # committed result JSONs (full-BAM / prior verified runs)
RESULTS = pathlib.Path(os.environ.get("ML4RG_RESULTS", COMMITTED))   # where live recomputes are written
def load_json(name, where=COMMITTED):
    p = pathlib.Path(where) / name
    return json.loads(p.read_text()) if p.exists() else None
print("repo:", REPO, "| committed results:", COMMITTED.exists())

In [ ]:
mj = load_json("mixcoeff_per_rbp.json")
assert mj is not None, "mixcoeff_per_rbp.json missing from committed results"
rows = [r for r in mj["rows"] if "mix_coeff" in r]
mc = np.array([r["mix_coeff"] for r in rows])
print(f"{len(rows)} RBP-cell tracks | mix_coeff mean={mc.mean():.3f} median={np.median(mc):.3f} "
      f"min={mc.min():.3f} max={mc.max():.3f}")
top = sorted(rows, key=lambda r:-r["mix_coeff"])[:8]
bot = sorted(rows, key=lambda r: r["mix_coeff"])[:8]
print("\nhighest (sequence-driven):")
for r in top: print(f"  {r['rbp']:9} {r.get('cell',''):5} family={r.get('family','?'):14} mix={r['mix_coeff']:.3f}")
print("\nlowest (bias-driven):")
for r in bot: print(f"  {r['rbp']:9} {r.get('cell',''):5} family={r.get('family','?'):14} mix={r['mix_coeff']:.3f}")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,3))
ax.hist(mc, bins=24, color="#4477aa", edgecolor="white")
ax.axvline(np.median(mc), color="#cc3311", ls="--", lw=1, label=f"median {np.median(mc):.2f}")
ax.set_xlabel("PARNET mix-coefficient (0 = bias-driven, 1 = sequence-driven)")
ax.set_ylabel("RBP-cell tracks"); ax.set_title("Per-RBP mix-coefficient distribution"); ax.legend()
fig.tight_layout(); out = REPO/"notebooks"/"demo"/"mixcoeff_hist.png"; fig.savefig(out, dpi=110)
print("saved", out)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(
f"""## Conclusion

Across **{len(rows)}** RBP-cell tracks the mix-coefficient spans **{mc.min():.2f}–{mc.max():.2f}**
(median **{np.median(mc):.2f}**): most tracks are strongly sequence-driven, with a bias-dominated tail.
The high-mix, motif-bearing RBPs are the candidates where a protein representation could plausibly help
(M1 group-selection TEST set); the low-mix tail are natural falsification controls. This interpretability
read is what links the demo to the milestone-1 deliverable. Claude-assisted; values from the committed
`mixcoeff_per_rbp.json` (frozen PARNET additive bias-mixture, Horlacher 2023)."""))